# Классификация жестов польского жестового языка (PJM)

## Постановка задачи

Задача: **многоклассовая классификация** жестов по координатам точек руки (hand landmarks),
полученных с помощью Google MediaPipe.

- **Вход**: координаты 21 точки кисти (x, y, z) × 3 фрейма = 63 признака
- **Выход**: класс жеста (буква/знак PJM)
- **Метрики**: Accuracy, F1-macro (macro важен при возможном дисбалансе классов)
- **Целевые значения**: Accuracy ≥ 85%, F1-macro ≥ 0.80

**Актуальность**: автоматическое распознавание жестового языка помогает
слабослышащим людям в коммуникации и может стать основой для систем перевода в реальном времени.
Принцип модели универсален и переносим на другие жестовые языки (РЖЯ, ASL и др.)
при наличии аналогичных данных.

## Импорты и настройка окружения

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import f1_score

from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.models import load_model

from utils import (
    load_data, encode_labels, normalize_landmarks,
    reshape_for_lstm, get_splits,
    build_bilstm,
    plot_training_history, plot_confusion_matrix, print_metrics
)

RANDOM_STATE = 42
DATA_PATH    = 'data/landmarks.csv'

## 1. Загрузка и разведочный анализ данных (EDA)

In [ ]:
df = load_data(DATA_PATH)
print(f"Размер датасета: {df.shape}")
print(f"Столбцы: {df.columns.tolist()[:5]} ...")
df.head()

In [ ]:
# Пропуски
print("Пропуски:\n", df.isnull().sum().sum())

# Статистика по координатам
df.describe()

In [ ]:
# Распределение классов
plt.figure(figsize=(14, 4))
df['label'].value_counts().sort_index().plot(kind='bar', color='steelblue')
plt.title('Количество примеров по классам')
plt.xlabel('Жест')
plt.ylabel('Количество')
plt.tight_layout()
plt.savefig('results/class_distribution.png', dpi=150)
plt.show()

In [ ]:
# Визуализация landmarks одного жеста (scatter plot руки)
sample = df[df['label'] == df['label'].iloc[0]].iloc[0]
coords = sample.drop('label').values.reshape(3, 21, 3)  # (frames, points, xyz)
frame  = coords[0]  # берём первый фрейм

plt.figure(figsize=(4, 5))
plt.scatter(frame[:, 0], -frame[:, 1], c=range(21), cmap='viridis', s=80)
for i, (x, y, _) in enumerate(frame):
    plt.text(x, -y + 0.01, str(i), fontsize=7)
plt.title(f"Landmarks — жест: '{sample['label']}'")
plt.axis('off')
plt.savefig('results/landmark_example.png', dpi=150)
plt.show()

## 2. Предобработка данных

In [ ]:
X_raw = df.drop('label', axis=1).values
y_raw, le = encode_labels(df, 'label')

n_classes   = len(le.classes_)
class_names = le.classes_
print(f"Количество классов: {n_classes}")
print(f"Классы: {class_names}")

In [ ]:
# Нормализация + reshape для LSTM и плоский вариант для baseline
X_scaled, scaler = normalize_landmarks(X_raw)

X_lstm = reshape_for_lstm(X_scaled, timesteps=3)   # (samples, 3, 63)
X_flat = X_scaled                                   # (samples, 189) — для RF/SVM

print(f"Форма для LSTM:     {X_lstm.shape}")
print(f"Форма для baseline: {X_flat.shape}")

In [ ]:
# Разбивка train / val / test  (70% / 15% / 15%)
X_train, X_val, X_test, y_train, y_val, y_test = get_splits(X_lstm, y_raw)
X_train_f, _, X_test_f, _, _, _ = get_splits(X_flat, y_raw)

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")

## 3. Baseline-модели (Random Forest, SVM)

Обучаем классические модели на плоском векторе признаков (без учёта временно́й структуры).
Это точка отсчёта для сравнения с BiLSTM.

In [ ]:
# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train_f, y_train)
y_pred_rf = rf.predict(X_test_f)

print("=== Random Forest ===")
f1_rf = print_metrics(y_test, y_pred_rf, class_names)
plot_confusion_matrix(y_test, y_pred_rf, class_names,
                      save_path='results/cm_random_forest.png')

In [ ]:
# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train_f, y_train)
y_pred_rf = rf.predict(X_test_f)

print("=== Random Forest ===")
f1_rf = print_metrics(y_test, y_pred_rf, class_names)
plot_confusion_matrix(y_test, y_pred_rf, class_names,
                      save_path='results/cm_random_forest.png')

In [ ]:
# SVM
svm = SVC(kernel='rbf', C=1.0, random_state=RANDOM_STATE)
svm.fit(X_train_f, y_train)
y_pred_svm = svm.predict(X_test_f)

print("=== SVM ===")
f1_svm = print_metrics(y_test, y_pred_svm, class_names)

## 4. Основная модель — Bidirectional LSTM

Используем временну́ю структуру данных: 3 фрейма как последовательность.
BiLSTM читает последовательность в обоих направлениях, что улучшает качество.

Архитектура: `Input → BiLSTM(64) → Dropout → Dense(128) → Dense(n_classes, softmax)`

In [ ]:
input_shape = (X_train.shape[1], X_train.shape[2])  # (3, 63)

model = build_bilstm(
    input_shape=input_shape,
    n_classes=n_classes,
    lstm_units=64,
    dropout_rate=0.3,
    learning_rate=0.001
)
model.summary()

In [ ]:
# Callbacks: остановка при отсутствии улучшений + сохранение лучших весов
callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ModelCheckpoint('results/best_model.keras', save_best_only=True)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# Графики обучения
plot_training_history(history, save_path='results/training_history.png')

In [ ]:
# Оценка на тестовой выборке
y_pred_lstm = model.predict(X_test).argmax(axis=1)

print("=== BiLSTM ===")
f1_lstm = print_metrics(y_test, y_pred_lstm, class_names)
plot_confusion_matrix(y_test, y_pred_lstm, class_names,
                      save_path='results/cm_bilstm.png')

## 5. Сравнение результатов

In [ ]:
results = pd.DataFrame([
    {'Модель': 'Random Forest (baseline)', 'F1-macro': round(f1_rf,   4)},
    {'Модель': 'SVM (baseline)',            'F1-macro': round(f1_svm,  4)},
    {'Модель': 'BiLSTM (наша модель)',      'F1-macro': round(f1_lstm, 4)},
])

# Выделяем лучший результат
print(results.to_string(index=False))
results.to_csv('results/metrics_report.csv', index=False)

In [ ]:
# Визуальное сравнение
plt.figure(figsize=(7, 4))
colors = ['#aec6cf', '#aec6cf', '#2196F3']
plt.barh(results['Модель'], results['F1-macro'], color=colors)
plt.xlim(0, 1.0)
plt.xlabel('F1-macro')
plt.title('Сравнение моделей')
plt.tight_layout()
plt.savefig('results/model_comparison.png', dpi=150)
plt.show()